In [5]:
import os

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI

openai_client = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=os.environ.get("GEMINI_API_KEY")
)

In [3]:
from IPython.display import Markdown

def llm(prompt):
    response = openai_client.chat.completions.create(
        model="gemini-3.5-flash",
        messages=[{"role": "user", "content": prompt}]
    )
    return Markdown(response.choices[0].message.content)

In [4]:
llm("Hey, what's up?")

Hey! Not much, just here and ready to help you out. How's your day going? What's on your mind?

In [5]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
answer

Welcome! I would love to help you get started, but I need a little more information first. 

Because I don't have context on **which specific course** you are referring to, here is how it usually works depending on the type of course:

1. **If it is a self-paced online course** (like on Udemy, Coursera, or a creator's website): **Yes, absolutely!** You can almost always enroll and start learning immediately.
2. **If it is a live, cohort-based course, bootcamp, or university class:** There might be specific registration deadlines, or you may have to wait for the next cohort to open.

**To help me give you the exact answer, could you tell me:**
* What is the name of the course?
* What platform or website is hosting it? (Or share the link if you have it!)

Once you share those details, I can guide you on how to join!

In [6]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [8]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [9]:
answer = llm(prompt)
answer

Yes, you can still join. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. You can also start learning and submitting homework (while the form is open) without registering.

In [12]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [13]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1342

In [14]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [15]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [16]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [18]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'When will the course be offered next?',
 'I missed the first homework - can I still get a certificate?']

In [25]:
index.search(
    question,
    num_results=5,
    boost_dict={"question": 2.0, "section": 0.5}
)

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section':

In [24]:
index.search(
    question,
    num_results=5,
    filter_dict={"course": "mlops-zoomcamp"}
)

[{'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course - Can I still join the course after the start date?',
  'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."},
 {'id': 'c842475338',
  'course': 'mlops-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Homework: Just found this course, can I still submit homeworks?',
  'answer': 'To clarify on **late homework submissions**:\n\n- You cannot submit after the homework is scored, as the form is closed.\n- Once the form is closed (i.e., scored), no further submissions are possible.\n- You can check your code against the solution by reviewing the `homework.md` file.\n\nIf the due date has passed but the form is still "O

In [26]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [34]:
search_results =search(question)

In [35]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [36]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [37]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [38]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [39]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [43]:
llm(prompt)

Yes, you can still join. However, if you want to receive a certificate, you must submit your project while project submissions are still being accepted. 

You can start learning and submitting homework (while the forms are open) even without receiving a registration confirmation, as the registration list is not checked for course participation.

In [44]:
response = openai_client.chat.completions.create(
        model="gemini-3.5-flash",
        messages=[{"role": "user", "content": prompt}]
    )

In [51]:
from pprint import pprint
pprint(response.to_dict())

{'choices': [{'finish_reason': 'stop',
              'index': 0,
              'message': {'content': 'Yes, you can still join! \n'
                                     '\n'
                                     'However, if you want to earn a '
                                     'certificate, you must submit your '
                                     'project while submissions are still '
                                     'being accepted. You can simply start '
                                     'learning and submitting homework (as '
                                     'long as the submission forms are still '
                                     'open) without needing to wait for a '
                                     'registration confirmation, as '
                                     'registration is not checked against any '
                                     'registered list.',
                          'extra_content': {'google': {'thought_signature': 'EtAOCs0OAQw5

In [58]:
def llm(instructions, user_prompt, model="gemini-3.5-flash"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.chat.completions.create(
        model=model,
        messages=message_history
    )

    return response.choices[0].message.content

In [59]:
llm(INSTRUCTIONS, prompt)

'Yes, you can still join. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

In [62]:
def rag(query, model="gemini-3.5-flash"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [63]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join. However, if you want to receive a certificate, you will need to submit your project while submissions are still being accepted.


In [64]:
rag("How do I get a certificate?")

'Based on the provided context, to get a certificate you must:\n\n1. **Finish the course with a "live" cohort:** Certificates are not awarded for the self-paced mode.\n2. **Pass the Capstone project:** While homework is not mandatory, passing the Capstone project is required.\n3. **Peer-review 3 capstones:** You must peer-review 3 capstone projects after submitting your project. This must be done while the course is actively running (after the form is closed and the peer-review list is compiled).\n\nAdditionally, to ensure your real name appears on the certificate instead of a randomly assigned nickname, you should update the second field in your "Edit Course Profile" with your official name (as it appears on your identification documents).'

In [8]:
from dotenv import load_dotenv
load_dotenv()

from ingest import load_faq_data, build_index
from rag_helper import RAGBase

documents = load_faq_data()
index = build_index(documents)

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    model="gemini-2.5-flash"
)

answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [9]:
assistant.rag("How do I get a certificate?")

'To get a certificate, you must:\n\n1.  **Finish the course with a "live" cohort.** Certificates are not awarded for self-paced mode.\n2.  **Pass the Capstone project.**\n3.  **Peer-review 3 other capstone projects** after submitting your own. This can only be done while the course is running.\n\nMissing homework will not prevent you from getting a certificate, as long as you pass the Capstone project.'

In [10]:
assistant.rag("Can I still join the course after it started?")

'Yes, you can still join the course even after it has started. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. You can start whenever you want as the videos and GitHub materials are available.'